#Design Query Embedding for searching shopping products

Example: "iphone charger"      ↔ "lightning cable"

"wireless earbuds"    ↔ "bluetooth earphones"

Prepare Training data using previous shopping data:

| Query            | Clicked Product       |
| ---------------- | --------------------- |
| iphone charger   | Apple Lightning Cable |
| iphone charger   | Anker Charger         |
| wireless earbuds | Sony Earbuds          |


Then, create training triplets:

Anchor Query      Positive Product        Negative Product

----------------------------------------------------------------


iphone charger    Apple Charger           Nike Shoes

iphone charger    Anker Charger           Samsung TV

wireless earbuds  Sony Earbuds            Coffee Mug


In [12]:
#iphone charger ↔ lightning cable = similar
#iphone charger ↔ running shoes = dissimilar

Training from scratch with Triplet Loss
Idea

We train using:

Anchor (A)
Positive (P)
Negative (N)

Example:

A = "iphone charger"
P = "lightning cable"
N = "running shoes"

We want:

distance(A,P) << distance(A,N)
Triplet Loss Formula
L=max(0,d(A,P)−d(A,N)+m)

Anchor ----- Positive

Anchor ------------------------ Negative

In [13]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn as nn

# -------------------------
# Model
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

encoder = AutoModel.from_pretrained(
    "bert-base-uncased"
)

triplet_loss = nn.TripletMarginLoss(
    margin=0.5
)

optimizer = torch.optim.Adam(
    encoder.parameters(),
    lr=1e-4
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:

# -------------------------
# Training data
# -------------------------
anchor_text = [
    "iphone charger",
    "wireless earbuds",
    "laptop sleeve",
    "coffee maker",
    "running shoes",
    "gaming keyboard",
    "usb cable",
    "water bottle"
]

positive_text = [
    "lightning cable",
    "bluetooth earphones",
    "macbook case",
    "espresso machine",
    "sports shoes",
    "mechanical keyboard",
    "charging cable",
    "insulated bottle"
]

negative_text = [
    "running shoes",
    "coffee mug",
    "dog food",
    "iphone case",
    "wireless mouse",
    "kitchen knife",
    "water bottle",
    "gaming monitor"
]

# -------------------------
# Tokenize once
# -------------------------
anchor_inputs = tokenizer(
    anchor_text,
    padding=True,
    truncation=True,
    return_tensors="pt"
)
print("Positive tokens are",anchor_inputs)

positive_inputs = tokenizer(
    positive_text,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

negative_inputs = tokenizer(
    negative_text,
    padding=True,
    truncation=True,
    return_tensors="pt"
)


Positive tokens are {'input_ids': tensor([[  101, 18059,  3715,  2099,   102,     0],
        [  101,  9949,  4540,  8569,  5104,   102],
        [  101, 12191, 10353,   102,     0,     0],
        [  101,  4157,  9338,   102,     0,     0],
        [  101,  2770,  6007,   102,     0,     0],
        [  101, 10355,  9019,   102,     0,     0],
        [  101, 18833,  5830,   102,     0,     0],
        [  101,  2300,  5835,   102,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0, 0]])}


In [15]:

# -------------------------
# Training loop
# -------------------------
for epoch in range(20):

    optimizer.zero_grad()

    anchor_emb = encoder(
        **anchor_inputs
    ).last_hidden_state[:, 0, :]

    positive_emb = encoder(
        **positive_inputs
    ).last_hidden_state[:, 0, :]

    negative_emb = encoder(
        **negative_inputs
    ).last_hidden_state[:, 0, :]

    loss = triplet_loss(
        anchor_emb,
        positive_emb,
        negative_emb
    )

    loss.backward()
    optimizer.step()

    d_pos = torch.norm(
        anchor_emb - positive_emb,
        dim=1
    ).mean()

    d_neg = torch.norm(
        anchor_emb - negative_emb,
        dim=1
    ).mean()

    print(
        f"Epoch {epoch+1:2d} | "
        f"Loss={loss.item():.4f} | "
        f"d_pos={d_pos.item():.4f} | "
        f"d_neg={d_neg.item():.4f}"
    )

Epoch  1 | Loss=0.0253 | d_pos=6.4006 | d_neg=8.2361
Epoch  2 | Loss=0.0203 | d_pos=7.7438 | d_neg=11.9639
Epoch  3 | Loss=0.0000 | d_pos=8.8017 | d_neg=15.6879
Epoch  4 | Loss=0.0154 | d_pos=9.2627 | d_neg=15.8053
Epoch  5 | Loss=0.0000 | d_pos=9.6082 | d_neg=15.7813
Epoch  6 | Loss=0.0000 | d_pos=8.6283 | d_neg=13.7425
Epoch  7 | Loss=0.1340 | d_pos=8.0847 | d_neg=12.7151
Epoch  8 | Loss=0.0000 | d_pos=7.8709 | d_neg=12.7904
Epoch  9 | Loss=0.0000 | d_pos=7.9712 | d_neg=12.8747
Epoch 10 | Loss=0.0000 | d_pos=8.0615 | d_neg=12.8991
Epoch 11 | Loss=0.0000 | d_pos=8.1093 | d_neg=12.8822
Epoch 12 | Loss=0.0000 | d_pos=8.1488 | d_neg=12.8426
Epoch 13 | Loss=0.0000 | d_pos=8.1846 | d_neg=12.7848
Epoch 14 | Loss=0.0000 | d_pos=8.2168 | d_neg=12.7137
Epoch 15 | Loss=0.0094 | d_pos=8.2463 | d_neg=12.6332
Epoch 16 | Loss=0.0000 | d_pos=8.3111 | d_neg=13.4697
Epoch 17 | Loss=0.0000 | d_pos=8.5877 | d_neg=13.5426
Epoch 18 | Loss=0.1245 | d_pos=8.6102 | d_neg=13.2695
Epoch 19 | Loss=0.0000 | d_po